# Milestone 4 — VLM Membership Inference Attack

In [ ]:
import sys
import os

# Ensure SAVE_DIR is defined before using it for sys.path.insert
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/milestone4/'

sys.path.insert(0, SAVE_DIR)

print(f'sys.path after insert: {sys.path}')
if os.path.exists(SAVE_DIR):
    print(f'Contents of {SAVE_DIR}: {os.listdir(SAVE_DIR)}')
else:
    print(f'{SAVE_DIR} does not exist.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sys.path after insert: ['/content/drive/MyDrive/milestone4/', '/content/drive/MyDrive/milestone4/', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
Contents of /content/drive/MyDrive/milestone4/: ['config.py', 'models.py', 'batches.py', 'classifier.py', 'features.py', '__pycache__']


In [ ]:
!pip install -q xgboost
!pip install num2words

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import torch

from models import load_data, load_model_and_processor
from batches import process_dataset_in_batches
from classifier import build_membership_classifier, generate_submission

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Always load splits — needed for labels and test IDs even when using cached features.
splits = load_data()

Device: cuda
Loading dataset: UBC-SLIME/colx585_group_project_data


Train:      6000 samples  (1500 members)
Validation: 1200 samples  (300 members)
Test:       6000 samples


## 1. Extract Features

Run the cells in this section to extract features from scratch.
Skip to **Section 2** if you already have cached CSVs on Drive.

In [ ]:
processor, model = load_model_and_processor(device)

Using dtype: torch.bfloat16
Loading model: UBC-SLIME/colx_585_vlm


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

Model loaded.


In [ ]:
train_features = process_dataset_in_batches(
    model, processor, splits['train'], has_labels=True, batch_size=200
)
train_features.to_csv(f'{SAVE_DIR}train.csv', index=False)
print('Train features saved.')

Processing 6000 samples in batches of 200...
  Batch 1/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:28<00:00,  7.03it/s]


  Batch 2/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.45it/s]


  Batch 3/30  (200 samples)


...

  Batch 29/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.59it/s]


  Batch 30/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.57it/s]


Done. 6000 samples processed.
Train features saved.


In [ ]:
val_features = process_dataset_in_batches(
    model, processor, splits['validation'], has_labels=True, batch_size=200
)
val_features.to_csv(f'{SAVE_DIR}val.csv', index=False)
print('Val features saved.')

Processing 1200 samples in batches of 200...
  Batch 1/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.57it/s]


  Batch 2/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.57it/s]


  Batch 3/6  (200 samples)


...

  Batch 5/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.55it/s]


  Batch 6/6  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.55it/s]

Done. 1200 samples processed.
Val features saved.


In [ ]:
test_features = process_dataset_in_batches(
    model, processor, splits['test'], has_labels=False, batch_size=200
)
test_features.to_csv(f'{SAVE_DIR}test.csv', index=False)
print('Test features saved.')

Processing 6000 samples in batches of 200...
  Batch 1/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.58it/s]


  Batch 2/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.49it/s]


  Batch 3/30  (200 samples)


...

  Batch 29/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.56it/s]


  Batch 30/30  (200 samples)


Extracting features: 100%|██████████| 200/200 [00:26<00:00,  7.58it/s]


Done. 6000 samples processed.
Test features saved.


## 2. Load Cached Features

Run this cell instead of Section 1 when resuming a session.

In [ ]:
# import pandas as pd
# train_features = pd.read_csv(f'{SAVE_DIR}train.csv')
# val_features   = pd.read_csv(f'{SAVE_DIR}val.csv')
# test_features  = pd.read_csv(f'{SAVE_DIR}test.csv')
# print('Features loaded.')


## 3. Train Classifier & Evaluate

In [ ]:
results = build_membership_classifier(train_features, val_features)

print('\n── Summary ──')
for name, r in results.items():
    print(f'{name:25s}  AUC={r["auc"]:.4f}  TPR@FPR=0.1={r["tpr_at_fpr01"]:.4f}')

best_name = max(results, key=lambda x: results[x]['auc'])
best      = results[best_name]
print(f'\nBest: {best_name}  AUC={best["auc"]:.4f}')

Features: 15  |  Train: 6000  |  Val: 1200

Training Logistic Regression...
  AUC=0.5096  TPR@FPR=0.1=0.1100  Acc=0.7500

Training Random Forest...
  AUC=0.4997  TPR@FPR=0.1=0.1033  Acc=0.7483
  Top-5: ['min_token_prob', 'loss', 'min_k10_prob', 'caption_loss', 'min_k20_prob']

Training XGBoost...
  AUC=0.5338  TPR@FPR=0.1=0.1533  Acc=0.7500
  Top-5: ['min_k20_prob', 'caption_loss', 'low_conf_ratio', 'min_k10_prob', 'min_k30_prob']

Training Gradient Boosting...
  AUC=0.5325  TPR@FPR=0.1=0.1500  Acc=0.7483
  Top-5: ['caption_loss', 'loss', 'min_k10_prob', 'low_conf_ratio', 'min_k30_prob']

Building weighted ensemble...
  Weights: XGBoost=0.257  Random Forest=0.241  Logistic Regression=0.246  Gradient Boosting=0.257
  AUC=0.5267  TPR@FPR=0.1=0.1300  Acc=0.7500

── Summary ──
Logistic Regression        AUC=0.5096  TPR@FPR=0.1=0.1100
Random Forest              AUC=0.4997  TPR@FPR=0.1=0.1033
XGBoost                    AUC=0.5338  TPR@FPR=0.1=0.1533
Gradient Boosting          AUC=0.5325  TPR

## 4. Generate Submission

In [ ]:
classifier_path = f'{SAVE_DIR}classifier.pkl'
with open(classifier_path, 'wb') as f:
    pickle.dump(best, f)
print(f'Classifier saved to {classifier_path}')

submission = generate_submission(
    splits['test'], test_features, best,
    output_path='submission.csv'
)
print(submission.head())

Classifier saved to /content/drive/MyDrive/milestone4/classifier.pkl
Submission saved to submission.csv  (6000 rows)
Score range: [0.0574, 0.6335]
                   id  is_member
0    cauldron_ln_8044   0.250106
1    cauldron_ln_7863   0.183169
2   mix_SI_UT_5400_19   0.149216
3  mix_SI_UT_5400_896   0.150046
4   cauldron_ln_17842   0.272893


In [ ]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>